# Notebook 05 — Memory System Demo

Demonstrates two memory layers:

1. **Session Memory (LangGraph SQLite Checkpointer)** — persist and resume individual conversations
2. **Cross-Session Semantic Memory (Mem0)** — extract facts from conversations and retrieve them in future sessions

References:
- Mem0: arxiv:2504.19413 (ECAI 2025) — +26% vs OpenAI Memory baseline
- LangGraph Persistence: langgraph-checkpoint-sqlite

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from memory.session_manager import get_checkpointer, new_thread_id
print('Memory modules loaded ✓')

## Part 1 — LangGraph Session Persistence (SQLite Checkpointer)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from tools import financial_tools
from rag import search_bank_policies

SYSTEM = 'You are SecureBank assistant. Be concise.'
all_tools = financial_tools + [search_bank_policies]
llm = ChatOllama(model='mistral', temperature=0)
llm_with_tools = llm.bind_tools(all_tools)

def agent_node(state):
    return {'messages': [llm_with_tools.invoke(state['messages'])]}

builder = StateGraph(MessagesState)
builder.add_node('agent', agent_node)
builder.add_node('tools', ToolNode(all_tools))
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', tools_condition)
builder.add_edge('tools', 'agent')

checkpointer = get_checkpointer()
app = builder.compile(checkpointer=checkpointer)
print('Persistent graph compiled ✓')

In [ ]:
# Session 1: first turn
thread_id = new_thread_id()
config = {'configurable': {'thread_id': thread_id}}
print(f'Session thread: {thread_id[:8]}...')

turn1 = [SystemMessage(content=SYSTEM), HumanMessage(content='What is the balance for CUST123?')]
result1 = app.invoke({'messages': turn1}, config=config)
print(f'Turn 1 answer: {result1["messages"][-1].content}')

In [ ]:
# Same session, second turn — agent remembers CUST123 was already discussed
turn2 = [HumanMessage(content='What about the mortgage rate for that account?')]
result2 = app.invoke({'messages': turn2}, config=config)
print(f'Turn 2 answer: {result2["messages"][-1].content}')

print(f'\nTotal messages in session: {len(result2["messages"])}')

In [ ]:
# Resume same session in a new Python context (simulating app restart)
# The thread_id is the only thing needed to restore full history
print(f'Resuming session {thread_id[:8]}... after simulated restart')

turn3 = [HumanMessage(content='What was the balance I asked about earlier?')]
result3 = app.invoke({'messages': turn3}, config=config)
print(f'Resume answer: {result3["messages"][-1].content}')
print('\n✓ Session correctly restored from SQLite checkpoint')

## Part 2 — Cross-Session Memory with Mem0

In [ ]:
from memory.session_manager import save_memory, load_memory, search_memory

# Simulate a conversation where user reveals preferences
user_id = 'demo_user_001'

conversation = [
    {'role': 'user',      'content': 'I prefer fixed-rate mortgages over variable.'},
    {'role': 'assistant', 'content': 'Noted. SecureBank offers a 5% fixed rate for 2026.'},
    {'role': 'user',      'content': 'My budget for a monthly payment is around $1,500.'},
    {'role': 'assistant', 'content': 'At 5% over 25 years, $1,500/month supports about $255,000 principal.'},
    {'role': 'user',      'content': 'I have concerns about overdraft fees.'},
    {'role': 'assistant', 'content': 'The $35 fee applies immediately. I recommend keeping a buffer.'},
]

print('Saving conversation to Mem0...')
save_memory(user_id, conversation)
print('Memory saved ✓')

In [ ]:
# Next session: load memories
print('Loading memories for new session...')
memories = load_memory(user_id)
print('Retrieved memories:')
print(memories if memories else '(No memories found — Mem0 may not be configured with Ollama yet)')

In [ ]:
# Search for specific memory
relevant = search_memory(user_id, 'mortgage preference')
print(f'Memory search (mortgage): {relevant if relevant else "(not available)"}')

In [ ]:
# Visualize the memory system architecture
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: SQLite session memory timeline
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 8)
ax1.axis('off')
ax1.set_title('Session Memory (SQLite Checkpointer)', fontsize=12, fontweight='bold')

turns = [
    (5, 7.0, 'Session Start\nthread_id generated',    '#ecf0f1'),
    (5, 5.5, 'Turn 1: Balance query\n→ get_account_balance()', '#dae8fc'),
    (5, 4.0, 'Turn 2: Follow-up\n→ Agent recalls Turn 1',     '#d5e8d4'),
    (5, 2.5, 'App Restart\n→ Restore from thread_id',         '#fdebd0'),
    (5, 1.0, 'Turn 3: Resumed\n→ Full history available',     '#d5e8d4'),
]
for x, y, text, color in turns:
    box = mpatches.FancyBboxPatch((x-2.5, y-0.5), 5, 1,
        boxstyle='round,pad=0.1', facecolor=color, edgecolor='#333', linewidth=1.2)
    ax1.add_patch(box)
    ax1.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
for i in range(len(turns)-1):
    ax1.annotate('', xy=(turns[i+1][0], turns[i+1][1]+0.5),
                 xytext=(turns[i][0], turns[i][1]-0.5),
                 arrowprops=dict(arrowstyle='->', color='#333'))

# Right: Mem0 cross-session memory flow
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.axis('off')
ax2.set_title('Cross-Session Memory (Mem0)', fontsize=12, fontweight='bold')

mem_steps = [
    (5, 7.0, 'Session 1\nUser reveals preferences',    '#ecf0f1'),
    (5, 5.5, 'Mem0 extracts facts\n"prefers fixed-rate"\n"budget $1,500/month"', '#fdebd0'),
    (5, 3.8, 'Facts stored in\nFAISS vector store',    '#dae8fc'),
    (5, 2.3, 'Session 2 (new day)\nMem0 injects memories\ninto system prompt',  '#d5e8d4'),
    (5, 0.8, 'Agent personalizes\nresponse from memory', '#d5e8d4'),
]
for x, y, text, color in mem_steps:
    box = mpatches.FancyBboxPatch((x-2.5, y-0.55), 5, 1.1,
        boxstyle='round,pad=0.1', facecolor=color, edgecolor='#333', linewidth=1.2)
    ax2.add_patch(box)
    ax2.text(x, y, text, ha='center', va='center', fontsize=8.5, fontweight='bold')
for i in range(len(mem_steps)-1):
    ax2.annotate('', xy=(mem_steps[i+1][0], mem_steps[i+1][1]+0.55),
                 xytext=(mem_steps[i][0], mem_steps[i][1]-0.55),
                 arrowprops=dict(arrowstyle='->', color='#333'))

plt.suptitle('SecureBank AI Agent — Memory System', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../notebooks/memory_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Memory architecture diagram saved ✓')